# Notebook 00: Architecture Overview -- Multimodal RAG for Video QA

## Purpose

This notebook provides the **high-level architecture** of our multimodal Retrieval-Augmented Generation (RAG) pipeline for answering questions about TV show episodes. The pipeline is designed around the **TVQA-Long** dataset, which contains thousands of multiple-choice questions grounded in specific temporal segments of TV episodes.

## What is Multimodal RAG for Video QA?

Traditional QA systems operate over static text. Video QA is harder because:

1. **Evidence is scattered** across long episodes (20-45 minutes of dialogue per episode).
2. **Temporal grounding** is critical -- the answer often depends on what happened in a specific 5-20 second window.
3. **Multiple modalities** (dialogue, visual, audio) provide complementary evidence.

Our approach uses a **text-first RAG pipeline** that retrieves relevant subtitle chunks, reranks them, and then generates answers with citations back to the source evidence. This is a strong baseline that can later be augmented with visual features.

## Why TVQA-Long?

- Large-scale: 15,253 validation questions across 6 popular TV shows
- Grounded: Each question has timestamp annotations linking it to a specific clip
- Challenging: Requires understanding dialogue context, character relationships, and plot progression
- Realistic: Based on real TV episodes with natural conversational language

## Setup and Imports

We begin by importing the standard libraries we need for data exploration. We use `pathlib` for cross-platform path handling, `json` for loading the annotation files, `pandas` for tabular summaries, and `collections.Counter` for frequency counting. No heavy ML libraries are needed for this overview notebook.

In [1]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd

# Project paths
PROJECT_ROOT = Path("/Users/nipun.batra/Downloads/ML/multimodal-rag-video-qa")
DATA_DIR = PROJECT_ROOT / "data" / "tvqa"
ANNOTATIONS_DIR = DATA_DIR / "annotations"
SUBTITLES_DIR = DATA_DIR / "subtitles"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Annotations: {list(ANNOTATIONS_DIR.glob('*.json'))}")
print(f"Subtitles archive: {list(SUBTITLES_DIR.glob('*'))}")

Project root: /Users/nipun.batra/Downloads/ML/multimodal-rag-video-qa
Data directory: /Users/nipun.batra/Downloads/ML/multimodal-rag-video-qa/data/tvqa
Annotations: [PosixPath('/Users/nipun.batra/Downloads/ML/multimodal-rag-video-qa/data/tvqa/annotations/tvqa_val_edited.json'), PosixPath('/Users/nipun.batra/Downloads/ML/multimodal-rag-video-qa/data/tvqa/annotations/tvqa_preprocessed_subtitles.json')]
Subtitles archive: [PosixPath('/Users/nipun.batra/Downloads/ML/multimodal-rag-video-qa/data/tvqa/subtitles/tvqa_subtitles.zip')]


---

## 1. Dataset Overview

We now load the two core annotation files:

1. **`tvqa_val_edited.json`** -- The validation question set, organized hierarchically: show -> season -> episode -> {questions, clips}. Each question has 5 answer choices (a0-a4), a correct answer index, a timestamp range, and a video clip reference.

2. **`tvqa_preprocessed_subtitles.json`** -- Pre-segmented subtitle clips (21,793 entries), where each clip contains a list of timestamped dialogue lines from a specific scene.

Loading these files lets us compute basic statistics: how many shows, seasons, episodes, and questions we are working with.

In [2]:
# Load validation questions (hierarchical: show -> season -> episode -> {questions, clips})
with open(ANNOTATIONS_DIR / "tvqa_val_edited.json") as f:
    val_data = json.load(f)

# Load preprocessed subtitle clips
with open(ANNOTATIONS_DIR / "tvqa_preprocessed_subtitles.json") as f:
    subtitle_clips = json.load(f)

print(f"Number of shows: {len(val_data)}")
print(f"Show names: {list(val_data.keys())}")
print(f"Number of subtitle clips: {len(subtitle_clips):,}")

Number of shows: 6
Show names: ['The Big Bang Theory', 'How I Met You Mother', 'Castle', 'Friends', 'House M.D.', "Grey's Anatomy"]
Number of subtitle clips: 21,793


### Counting Questions, Seasons, and Episodes per Show

We traverse the hierarchical structure to count the total number of questions, seasons, and episodes per show. This gives us a sense of the dataset's scale and balance across different TV series.

In [3]:
# Compute per-show statistics
stats = []
total_questions = 0
total_episodes = 0
total_seasons = 0

for show_name, seasons in val_data.items():
    show_questions = 0
    show_episodes = 0
    show_seasons = len(seasons)
    total_seasons += show_seasons
    
    for season_name, episodes in seasons.items():
        for episode_name, episode_data in episodes.items():
            show_episodes += 1
            total_episodes += 1
            num_q = len(episode_data["questions"])
            show_questions += num_q
            total_questions += num_q
    
    stats.append({
        "Show": show_name,
        "Seasons": show_seasons,
        "Episodes": show_episodes,
        "Questions": show_questions
    })

stats_df = pd.DataFrame(stats)
stats_df = stats_df.sort_values("Questions", ascending=False).reset_index(drop=True)

print("=" * 70)
print("TVQA-Long Validation Set: Per-Show Statistics")
print("=" * 70)
print(stats_df.to_string(index=False))
print("-" * 70)
print(f"TOTALS: {total_seasons} seasons, {total_episodes} episodes, {total_questions:,} questions")
print(f"Subtitle clips available: {len(subtitle_clips):,}")

TVQA-Long Validation Set: Per-Show Statistics
                Show  Seasons  Episodes  Questions
             Friends       10       214       3920
              Castle        8       163       3311
          House M.D.        8       164       3234
 The Big Bang Theory       10       188       3017
How I Met You Mother        5        66       1043
      Grey's Anatomy        3        47        728
----------------------------------------------------------------------
TOTALS: 44 seasons, 842 episodes, 15,253 questions
Subtitle clips available: 21,793


### Interpreting Dataset Statistics

The table above reveals the scale and composition of our evaluation data:

- **6 TV shows** spanning different genres (sitcom, drama, procedural, medical)
- The dataset is spread across multiple seasons and episodes per show
- Each episode contributes roughly 10-30 questions on average
- The total of ~15,000 questions provides a statistically robust evaluation set

**Key observation:** The dataset is large enough that even per-show accuracy differences of 2-3 percentage points will be meaningful. The diversity of shows also tests whether our retrieval pipeline generalizes across different dialogue styles (fast-paced sitcom banter vs. medical jargon vs. detective procedurals).

### Sample Question Structure

Let us examine a single question to understand the data format that our pipeline must handle. Each question includes the question text, 5 candidate answers, the correct answer index, a timestamp range (in seconds), and a reference to the source video clip.

In [4]:
# Display a sample question
first_show = list(val_data.keys())[0]
first_season = list(val_data[first_show].keys())[0]
first_episode = list(val_data[first_show][first_season].keys())[0]
sample_question = val_data[first_show][first_season][first_episode]["questions"][0]

print(f"Show: {first_show}")
print(f"Season: {first_season}, Episode: {first_episode}")
print("-" * 70)
print(f"Question (qid={sample_question['qid']}):")
print(f"  {sample_question['q']}")
print()
print("Answer choices:")
for i in range(5):
    marker = " --> CORRECT" if i == sample_question['answer_idx'] else ""
    print(f"  (a{i}) {sample_question[f'a{i}']}{marker}")
print()
print(f"Timestamp: {sample_question['ts']} seconds")
print(f"Video clip: {sample_question['vid_name']}")

Show: The Big Bang Theory
Season: season_3, Episode: episode_2
----------------------------------------------------------------------
Question (qid=122039):
  Why is Howard frustrated when he is talking to Sheldon?

Answer choices:
  (a0) Because Sheldon is being rude.
  (a1) Because he doesn't like Sheldon.
  (a2) Because they are having an argument. --> CORRECT
  (a3) Because Howard wanted to have a private meal with Raj.
  (a4) Because Sheldon won't loan him money for food.

Timestamp: 20.16-25.12 seconds
Video clip: s03e02_seg02_clip_10


### Sample Subtitle Clip Structure

Each subtitle clip in the preprocessed file contains a `vid_name` (matching the video clip reference in questions) and a `sub` list of timestamped dialogue lines. This is the raw text evidence that our retrieval pipeline will index and search over.

In [5]:
# Display a sample subtitle clip
sample_clip = subtitle_clips[0]
print(f"Clip ID: {sample_clip['vid_name']}")
print(f"Number of dialogue lines: {len(sample_clip['sub'])}")
print("-" * 70)
print("First 5 dialogue lines:")
for line in sample_clip['sub'][:5]:
    print(f"  [{line['start']:.1f}s - {line['end']:.1f}s] {line['text'].strip()}")

# Also show distribution of lines per clip
lines_per_clip = [len(clip['sub']) for clip in subtitle_clips]
print()
print(f"Lines per clip -- min: {min(lines_per_clip)}, max: {max(lines_per_clip)}, "
      f"median: {sorted(lines_per_clip)[len(lines_per_clip)//2]}, "
      f"mean: {sum(lines_per_clip)/len(lines_per_clip):.1f}")

Clip ID: house_s02e05_seg02_clip_11
Number of dialogue lines: 40
----------------------------------------------------------------------
First 5 dialogue lines:
  [0.9s - 1.9s] Chase : That's all this is?
  [2.0s - 2.9s] Yeah.
  [3.0s - 5.3s] House : Because his white blood cell count was down, he was vulnerable.
  [5.4s - 7.1s] House : Because it's really down, it might kill him.
  [7.1s - 8.7s] Chase : That's all this is.

Lines per clip -- min: 1, max: 63, median: 22, mean: 22.6


### Interpreting the Subtitle Data

The subtitle clips provide the **text-based evidence** for our RAG pipeline:

- Each clip corresponds to a scene segment (typically 20-60 seconds of screen time)
- Dialogue lines include speaker identification (e.g., "Chase:", "House:") which is valuable for character-based questions
- Timestamps within each clip enable fine-grained temporal alignment with questions

**Key design implication:** Since clips vary in length, our chunking strategy must handle both very short clips (a few lines) and very long ones. We will need to decide whether to index at the clip level or sub-divide long clips into smaller retrieval units.

---

## 2. Pipeline Architecture Diagram

The following text-based diagram shows the end-to-end flow of our multimodal RAG pipeline. The pipeline has four major stages, each handled by dedicated notebooks:

1. **Text Retrieval** -- Index subtitle chunks and retrieve candidates matching a query
2. **Reranking** -- Score and reorder retrieved candidates by relevance
3. **Answer Generation** -- Use retrieved evidence to select the best answer with citations
4. **Evaluation** -- Measure accuracy, faithfulness, hallucination, and calibration

We display this as a structured text diagram so the architecture is immediately visible without external tools.

In [6]:
def print_architecture_diagram():
    """Print the full RAG pipeline architecture as a text diagram."""
    
    diagram = """
================================================================================
           MULTIMODAL RAG PIPELINE FOR VIDEO QA -- ARCHITECTURE
================================================================================

 INPUT DATA                        INDEXING PHASE
 ----------                        --------------
                                   
 +---------------------+          +---------------------------+
 | Episode Subtitles   |          | 1. Parse SRT files        |
 | (1,465 SRT files)   | -------> | 2. Chunk into passages    |
 +---------------------+          | 3. Build BM25 index       |
                                   +---------------------------+
 +---------------------+                      |
 | Preprocessed Clips  |                      |
 | (21,793 clips)      | --+                  v
 +---------------------+   |     +---------------------------+
                            |     |    BM25 Sparse Index      |
                            +---> |    (term frequencies,     |
                                  |     IDF weights)          |
                                  +---------------------------+


 QUERY PHASE (per question)
 --------------------------

 +---------------------+     +------------------+     +-------------------+
 | Question + Choices  |     | STAGE 1:         |     | STAGE 2:          |
 | "Why is Howard      | --> | Text Retrieval   | --> | Reranking          |
 |  frustrated...?"    |     | (BM25 top-k)     |     | (token overlap)   |
 +---------------------+     +------------------+     +-------------------+
                                                              |
                                                              v
                              +------------------+     +-------------------+
                              | STAGE 4:         |     | STAGE 3:          |
                              | Evaluation       | <-- | Answer Generation |
                              | - Accuracy       |     | - Evidence concat |
                              | - Faithfulness   |     | - MC selection    |
                              | - Hallucination  |     | - Citation attach |
                              | - Calibration    |     +-------------------+
                              +------------------+


 DETAILED STAGE BREAKDOWN
 ------------------------

 STAGE 1: TEXT RETRIEVAL
 +-----------------------------------------------------------------+
 | Input:  Question text (+ optionally concatenated answer choices)|
 | Method: BM25 sparse retrieval over chunked subtitle passages    |
 | Output: Top-k candidate passages (k=10 to 50)                  |
 | Key decisions:                                                   |
 |   - Chunk size (number of subtitle lines per passage)           |
 |   - Query formulation (question only vs. question + answers)    |
 |   - Retrieval scope (episode-level vs. full corpus)             |
 +-----------------------------------------------------------------+

 STAGE 2: RERANKING
 +-----------------------------------------------------------------+
 | Input:  Top-k BM25 candidates + original question               |
 | Method: Token overlap reranking (unigram/bigram matching)       |
 | Output: Reranked top-n passages (n <= k)                        |
 | Key decisions:                                                   |
 |   - Scoring function (Jaccard, weighted overlap, etc.)          |
 |   - How many candidates to keep after reranking                 |
 |   - Whether to include answer tokens in matching                |
 +-----------------------------------------------------------------+

 STAGE 3: ANSWER GENERATION
 +-----------------------------------------------------------------+
 | Input:  Reranked evidence passages + question + 5 answer choices|
 | Method: Evidence-based multiple choice selection with citations  |
 | Output: Selected answer index + supporting evidence citations   |
 | Key decisions:                                                   |
 |   - How to aggregate evidence across multiple passages          |
 |   - Confidence scoring for selective prediction                 |
 |   - Citation format and granularity                             |
 +-----------------------------------------------------------------+

 STAGE 4: EVALUATION
 +-----------------------------------------------------------------+
 | Input:  Predicted answers + ground truth + evidence citations    |
 | Metrics:                                                         |
 |   - Accuracy: fraction of correctly answered questions          |
 |   - Faithfulness: are citations relevant to the answer?         |
 |   - Hallucination: does the system cite non-existent evidence?  |
 |   - Selective prediction: accuracy when system is confident     |
 +-----------------------------------------------------------------+

================================================================================
"""
    print(diagram)

print_architecture_diagram()


           MULTIMODAL RAG PIPELINE FOR VIDEO QA -- ARCHITECTURE

 INPUT DATA                        INDEXING PHASE
 ----------                        --------------

 +---------------------+          +---------------------------+
 | Episode Subtitles   |          | 1. Parse SRT files        |
 | (1,465 SRT files)   | -------> | 2. Chunk into passages    |
 +---------------------+          | 3. Build BM25 index       |
                                   +---------------------------+
 +---------------------+                      |
 | Preprocessed Clips  |                      |
 | (21,793 clips)      | --+                  v
 +---------------------+   |     +---------------------------+
                            |     |    BM25 Sparse Index      |
                            +---> |    (term frequencies,     |
                                  |     IDF weights)          |
                                  +---------------------------+


 QUERY PHASE (per question)
 ------------------

### Architecture Design Rationale

The pipeline above follows a deliberate **retrieve-then-read** paradigm:

- **Why BM25 first?** BM25 is fast (milliseconds per query), requires no GPU, and provides strong baselines for text matching. It handles the "needle in a haystack" problem of finding the relevant 5-second clip from 45 minutes of dialogue.

- **Why reranking?** BM25 uses exact token matching and can miss semantic relevance. A reranking stage with token overlap scoring provides a lightweight improvement without requiring a neural model.

- **Why evidence-based generation?** By forcing the system to cite specific passages, we can measure faithfulness and detect hallucinations -- critical for trustworthy QA systems.

- **Why selective prediction?** Not all questions are equally answerable from text alone. A calibration mechanism lets the system abstain when uncertain, trading coverage for higher precision.

---

## 3. Component Dependency Map

This section shows which components depend on which others. Understanding dependencies is crucial for:
- Knowing which notebooks must be run in order
- Identifying which components can be developed in parallel
- Debugging failures (if Stage 3 fails, is the issue in retrieval or generation?)

We represent this as a structured table showing inputs, outputs, and dependencies for each component.

In [7]:
# Component dependency map
components = [
    {
        "Component": "Data Loading & EDA",
        "Inputs": "Raw JSON annotations, SRT files",
        "Outputs": "Cleaned DataFrames, show/episode indices",
        "Depends On": "(none -- entry point)",
    },
    {
        "Component": "Subtitle Parsing",
        "Inputs": "SRT files (1,465 episodes)",
        "Outputs": "Parsed subtitle records with timestamps",
        "Depends On": "Data Loading",
    },
    {
        "Component": "Chunking Strategy",
        "Inputs": "Parsed subtitles",
        "Outputs": "Chunked passages (fixed-size or semantic)",
        "Depends On": "Subtitle Parsing",
    },
    {
        "Component": "BM25 Indexing",
        "Inputs": "Chunked passages",
        "Outputs": "BM25 index (term freq, doc freq, IDF)",
        "Depends On": "Chunking Strategy",
    },
    {
        "Component": "Retrieval",
        "Inputs": "BM25 index + question query",
        "Outputs": "Top-k candidate passages per question",
        "Depends On": "BM25 Indexing",
    },
    {
        "Component": "Reranking",
        "Inputs": "Top-k candidates + question",
        "Outputs": "Reranked top-n passages",
        "Depends On": "Retrieval",
    },
    {
        "Component": "Answer Generation",
        "Inputs": "Reranked passages + question + choices",
        "Outputs": "Predicted answer + confidence + citations",
        "Depends On": "Reranking",
    },
    {
        "Component": "Evaluation",
        "Inputs": "Predictions + ground truth",
        "Outputs": "Accuracy, faithfulness, calibration metrics",
        "Depends On": "Answer Generation",
    },
]

dep_df = pd.DataFrame(components)
print("=" * 90)
print("COMPONENT DEPENDENCY MAP")
print("=" * 90)
print()
for i, row in dep_df.iterrows():
    print(f"[{i+1}] {row['Component']}")
    print(f"    Inputs:     {row['Inputs']}")
    print(f"    Outputs:    {row['Outputs']}")
    print(f"    Depends On: {row['Depends On']}")
    if i < len(dep_df) - 1:
        print(f"        |")
        print(f"        v")
print()
print("=" * 90)

COMPONENT DEPENDENCY MAP

[1] Data Loading & EDA
    Inputs:     Raw JSON annotations, SRT files
    Outputs:    Cleaned DataFrames, show/episode indices
    Depends On: (none -- entry point)
        |
        v
[2] Subtitle Parsing
    Inputs:     SRT files (1,465 episodes)
    Outputs:    Parsed subtitle records with timestamps
    Depends On: Data Loading
        |
        v
[3] Chunking Strategy
    Inputs:     Parsed subtitles
    Outputs:    Chunked passages (fixed-size or semantic)
    Depends On: Subtitle Parsing
        |
        v
[4] BM25 Indexing
    Inputs:     Chunked passages
    Outputs:    BM25 index (term freq, doc freq, IDF)
    Depends On: Chunking Strategy
        |
        v
[5] Retrieval
    Inputs:     BM25 index + question query
    Outputs:    Top-k candidate passages per question
    Depends On: BM25 Indexing
        |
        v
[6] Reranking
    Inputs:     Top-k candidates + question
    Outputs:    Reranked top-n passages
    Depends On: Retrieval
        

### Dependency Map Interpretation

The pipeline is **strictly sequential** -- each stage requires the output of the previous one. This means:

- **Development order matters:** We cannot evaluate answer generation without first having a working retrieval stage.
- **Error propagation is a concern:** If BM25 retrieval fails to surface the relevant passage, no amount of reranking or generation quality can recover. This motivates measuring **recall@k** at each stage.
- **Parallelism is limited to evaluation variants:** Once we have predictions, we can compute different metrics in parallel.

**Key insight:** The most impactful place to invest effort is the retrieval stage. If the correct evidence is not in the top-k candidates, the downstream pipeline cannot succeed. We will track retrieval recall closely.

---

## 4. Data Flow Summary

Before we proceed to the notebook roadmap, let us verify the key numbers that define the scale of our pipeline. These numbers will recur throughout the project as sanity checks.

In [8]:
# Count unique shows represented in subtitle clips
import re

clip_shows = Counter()
for clip in subtitle_clips:
    # vid_name format varies:
    #   - Most shows: showname_s##e##_seg##_clip_## (e.g., "house_s02e05_seg02_clip_11")
    #   - Big Bang Theory: s##e##_seg##_clip_## (no prefix, starts directly with season)
    vid = clip['vid_name']
    match = re.search(r'_s\d+e\d+_', vid)
    if match:
        show_prefix = vid[:match.start()]
        clip_shows[show_prefix] += 1
    elif re.match(r's\d+e\d+_', vid):
        # No prefix means Big Bang Theory
        clip_shows["bbt (no prefix)"] += 1

# Summary statistics
print("=" * 70)
print("DATA FLOW SUMMARY -- KEY NUMBERS")
print("=" * 70)
print()
print(f"  TV Shows:                  {len(val_data)}")
print(f"  Total Seasons:             {total_seasons}")
print(f"  Total Episodes:            {total_episodes}")
print(f"  Validation Questions:      {total_questions:,}")
print(f"  Answer Choices per Q:      5 (multiple choice)")
print(f"  Preprocessed Sub Clips:    {len(subtitle_clips):,}")
print(f"  SRT Files (archived):      1,465")
print()
print("  Shows in subtitle clips (by vid_name prefix):")
for show, count in clip_shows.most_common():
    print(f"    {show:30s} {count:,} clips")
print(f"    {'TOTAL':30s} {sum(clip_shows.values()):,} clips")
print()
print(f"  Random baseline accuracy:  {1/5:.1%} (5-choice MC)")
print("=" * 70)

DATA FLOW SUMMARY -- KEY NUMBERS

  TV Shows:                  6
  Total Seasons:             44
  Total Episodes:            842
  Validation Questions:      15,253
  Answer Choices per Q:      5 (multiple choice)
  Preprocessed Sub Clips:    21,793
  SRT Files (archived):      1,465

  Shows in subtitle clips (by vid_name prefix):
    friends                        4,870 clips
    castle                         4,698 clips
    house                          4,621 clips
    bbt (no prefix)                4,198 clips
    met                            1,512 clips
    grey                           1,427 clips
    TOTAL                          21,326 clips

  Random baseline accuracy:  20.0% (5-choice MC)


### Scale Interpretation

These numbers define the computational constraints of our pipeline:

- **15,000+ questions** means we need efficient retrieval -- a brute-force approach scanning all clips per question would require ~15,000 x 21,793 = 327 million comparisons. BM25 with inverted indices reduces this dramatically.
- **21,793 subtitle clips** is a manageable corpus for in-memory BM25 (each clip is a few hundred tokens at most).
- **Random baseline is 20%** (1 in 5). Any system that cannot beat this is performing worse than random guessing. A strong text-retrieval baseline should achieve 40-60% on this task.
- The clip distribution across shows will be important for per-show evaluation -- if one show has far fewer clips, retrieval may be harder for that show.

---

## 5. Notebook Roadmap (Notebooks 01-12)

The following table summarizes what each subsequent notebook in this series will cover. The notebooks are designed to be run in order, with each building on the outputs of the previous ones.

In [9]:
# Notebook roadmap
roadmap = [
    {
        "Notebook": "01",
        "Title": "Data Loading and EDA",
        "Focus": "Load all data files, explore distributions, validate data quality",
        "Key Output": "Clean DataFrames for questions and subtitles",
    },
    {
        "Notebook": "02",
        "Title": "Subtitle Parsing and Preprocessing",
        "Focus": "Parse SRT files, normalize text, handle speaker identification",
        "Key Output": "Unified subtitle corpus with metadata",
    },
    {
        "Notebook": "03",
        "Title": "Chunking Strategies",
        "Focus": "Experiment with fixed-size vs. semantic chunking approaches",
        "Key Output": "Chunked passages ready for indexing",
    },
    {
        "Notebook": "04",
        "Title": "BM25 Index Construction",
        "Focus": "Build BM25 index, tune parameters (k1, b), measure index stats",
        "Key Output": "Serialized BM25 index for fast retrieval",
    },
    {
        "Notebook": "05",
        "Title": "Retrieval Evaluation",
        "Focus": "Measure recall@k, MRR, per-show retrieval quality",
        "Key Output": "Retrieval metrics and error analysis",
    },
    {
        "Notebook": "06",
        "Title": "Token Overlap Reranking",
        "Focus": "Implement and evaluate reranking with token matching",
        "Key Output": "Reranked candidates with improved precision",
    },
    {
        "Notebook": "07",
        "Title": "Answer Generation (Baseline)",
        "Focus": "Simple heuristic answer selection from retrieved evidence",
        "Key Output": "Baseline predictions for all val questions",
    },
    {
        "Notebook": "08",
        "Title": "Answer Generation (LLM-based)",
        "Focus": "LLM-based evidence reading and answer selection",
        "Key Output": "LLM predictions with confidence scores",
    },
    {
        "Notebook": "09",
        "Title": "Citation and Evidence Tracking",
        "Focus": "Attach source citations to predictions, trace evidence",
        "Key Output": "Predictions with provenance metadata",
    },
    {
        "Notebook": "10",
        "Title": "Evaluation: Accuracy and Error Analysis",
        "Focus": "Overall and per-show accuracy, error categorization",
        "Key Output": "Accuracy metrics and error taxonomy",
    },
    {
        "Notebook": "11",
        "Title": "Evaluation: Faithfulness and Hallucination",
        "Focus": "Measure citation quality, detect unsupported claims",
        "Key Output": "Faithfulness scores and hallucination rates",
    },
    {
        "Notebook": "12",
        "Title": "Selective Prediction and Calibration",
        "Focus": "Confidence thresholding, coverage-accuracy tradeoff",
        "Key Output": "Calibration curves and optimal threshold",
    },
]

roadmap_df = pd.DataFrame(roadmap)

print("=" * 90)
print("NOTEBOOK ROADMAP: Multimodal RAG for Video QA (TVQA-Long)")
print("=" * 90)
print()
for _, row in roadmap_df.iterrows():
    print(f"  [{row['Notebook']}] {row['Title']}")
    print(f"      Focus:      {row['Focus']}")
    print(f"      Key Output: {row['Key Output']}")
    print()
print("=" * 90)
print(f"Total notebooks in series: {len(roadmap) + 1} (including this overview)")
print("=" * 90)

NOTEBOOK ROADMAP: Multimodal RAG for Video QA (TVQA-Long)

  [01] Data Loading and EDA
      Focus:      Load all data files, explore distributions, validate data quality
      Key Output: Clean DataFrames for questions and subtitles

  [02] Subtitle Parsing and Preprocessing
      Focus:      Parse SRT files, normalize text, handle speaker identification
      Key Output: Unified subtitle corpus with metadata

  [03] Chunking Strategies
      Focus:      Experiment with fixed-size vs. semantic chunking approaches
      Key Output: Chunked passages ready for indexing

  [04] BM25 Index Construction
      Focus:      Build BM25 index, tune parameters (k1, b), measure index stats
      Key Output: Serialized BM25 index for fast retrieval

  [05] Retrieval Evaluation
      Focus:      Measure recall@k, MRR, per-show retrieval quality
      Key Output: Retrieval metrics and error analysis

  [06] Token Overlap Reranking
      Focus:      Implement and evaluate reranking with token matching

### Roadmap Design Principles

The notebook sequence follows three principles:

1. **Data before models:** Notebooks 01-03 focus entirely on understanding and preparing the data. Premature modeling without understanding the data is the most common source of bugs in RAG pipelines.

2. **Incremental complexity:** We start with BM25 (the simplest effective retrieval method), add reranking, then move to LLM-based generation. Each step is independently evaluable.

3. **Evaluation throughout:** Rather than building the full pipeline and evaluating at the end, we measure retrieval quality (notebook 05) before adding generation. This prevents the common failure mode of optimizing generation when retrieval is the bottleneck.

---

## 6. Summary and Next Steps

This notebook established:

- **The problem:** Answering multiple-choice questions about TV episodes using subtitle-based evidence retrieval.
- **The data:** 6 shows, ~15,000 questions, ~22,000 subtitle clips, 1,465 full episode SRT files.
- **The architecture:** A 4-stage pipeline (Retrieve -> Rerank -> Generate -> Evaluate) built on BM25 sparse retrieval with token overlap reranking.
- **The roadmap:** 12 subsequent notebooks covering data preparation through calibrated evaluation.

**Next step:** Proceed to Notebook 01 (Data Loading and EDA) to deeply explore the question distributions, answer patterns, and subtitle characteristics that will inform our chunking and retrieval design decisions.